<a href="https://colab.research.google.com/github/Nadiaesterlianasiahaa/RET503-Computer-Vision-and-Deep-Learning-/blob/main/p1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q ultralytics
from ultralytics import YOLO
import ultralytics
ultralytics.checks()

from pathlib import Path
import zipfile, shutil, time, yaml
import pandas as pd

WORK = Path("/content/pj1")
WORK.mkdir(exist_ok=True)

from google.colab import files
print("Upload dataset_v1.zip")
uploaded = files.upload()

for name in uploaded:
    if name.endswith(".zip"):
        with zipfile.ZipFile(name) as z:
            z.extractall(WORK)
        print("OK:", name)

# cari folder dataset_v1
cands = list(WORK.rglob("data.yaml"))
assert cands, "data.yaml tidak ketemu"
DATA_DIR = cands[0].parent
print("DATA_DIR =", DATA_DIR)

Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 42.4/112.6 GB disk)
Upload dataset_v1.zip


Saving dataset_v1.zip to dataset_v1.zip
OK: dataset_v1.zip
DATA_DIR = /content/pj1/dataset_v1


In [2]:
yaml_path = DATA_DIR / "data.yaml"
cfg = yaml.safe_load(yaml_path.read_text())

# path absolut di Colab
cfg["path"] = str(DATA_DIR.resolve())
cfg["train"] = "images/train"
cfg["val"] = "images/val"
cfg["test"] = "images/test"

# pastikan nc ada
if "nc" not in cfg:
    cfg["nc"] = len(cfg["names"])

yaml_path.write_text(yaml.dump(cfg, sort_keys=False, allow_unicode=True))
print(yaml_path.read_text())

# verifikasi jumlah file
for split in ["train", "val", "test"]:
    ni = len(list((DATA_DIR / "images" / split).glob("*")))
    nl = len(list((DATA_DIR / "labels" / split).glob("*.txt")))
    print(f"{split:5s}  images={ni:4d}  labels={nl:4d}")

path: /content/pj1/dataset_v1
train: images/train
val: images/val
test: images/test
names:
  0: Blue 3
  1: Blue 8
  2: Blue D
  3: Blue Left Arrow
  4: Blue S
  5: Blue X
  6: Green 2
  7: Green 7
  8: Green C
  9: Green H
  10: Green Right Arrow
  11: Green W
  12: Red 4
  13: Red 9
  14: Red Arrow Down
  15: Red E
  16: Red T
  17: Red Y
  18: White 1
  19: White 6
  20: White Arrow Up
  21: White B
  22: White Bullseye
  23: White G
  24: White V
  25: Yellow 5
  26: Yellow A
  27: Yellow Circle
  28: Yellow F
  29: Yellow U
  30: Yellow Z
nc: 31

train  images= 972  labels= 972
val    images= 278  labels= 278
test   images= 140  labels= 140


In [3]:
from collections import Counter

bad, boxes, ids = [], 0, []
for p in (DATA_DIR / "labels").rglob("*.txt"):
    lines = [ln for ln in p.read_text().splitlines() if ln.strip()]
    for ln in lines:
        parts = ln.split()
        if len(parts) != 5:
            bad.append((p.name, ln, "bukan_5_kolom")); continue
        try:
            cid = int(float(parts[0]))
            vals = list(map(float, parts[1:]))
        except Exception:
            bad.append((p.name, ln, "parse")); continue
        if not (0 <= cid < 31):
            bad.append((p.name, ln, "class_id")); continue
        if not all(0 <= v <= 1 for v in vals):
            bad.append((p.name, ln, "norm")); continue
        ids.append(cid); boxes += 1

print("Total box:", boxes)
print("Unique class:", len(set(ids)), "/ 31")
print("Bad lines:", len(bad))
if bad[:5]:
    print("Contoh error:", bad[:5])
else:
    print("Anotasi OK — siap train.")

Total box: 1390
Unique class: 31 / 31
Bad lines: 0
Anotasi OK — siap train.


In [5]:
DATA_YAML = str(DATA_DIR / "data.yaml")
PROJ = str(WORK / "hasil_praktik")

model = YOLO("yolo26n.pt")  # fallback: YOLO("yolov8n.pt")
model.train(
    data=DATA_YAML,
    epochs=10,
    imgsz=416,
    batch=16,
    project=PROJ,
    name="E1_pretrained",
    exist_ok=True,
)
print("E1 selesai →", PROJ + "/E1_pretrained")

Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/pj1/dataset_v1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=E1_pretrained, nbs=64, nms=None, opset=None, optimize=Fal

In [6]:
# Arsitektur sama, bobot acak
model = YOLO("yolo26n.yaml")  # atau "yolov8n.yaml"
model.train(
    data=DATA_YAML,
    epochs=10,
    imgsz=416,
    batch=16,
    pretrained=False,
    project=PROJ,
    name="E2_scratch",
    exist_ok=True,
)
print("E2 selesai")

Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/pj1/dataset_v1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=E2_scratch, nbs=64, nms=None, opset=None, optimize=Fals

In [7]:
model = YOLO("yolo26n.pt")
model.train(
    data=DATA_YAML,
    epochs=10,
    imgsz=640,
    batch=16,
    project=PROJ,
    name="E3_tuning",
    exist_ok=True,
)
print("E3 selesai")

Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/pj1/dataset_v1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=E3_tuning, nbs=64, nms=None, opset=None, optimize=False, 

In [8]:
rows = []
for name, imgsz in [
    ("E1_pretrained", 416),
    ("E2_scratch", 416),
    ("E3_tuning", 640),
]:
    w = WORK / "hasil_praktik" / name / "weights" / "best.pt"
    if not w.exists():
        print("SKIP belum ada:", w)
        continue
    m = YOLO(str(w))
    met = m.val(data=DATA_YAML, split="val", imgsz=imgsz, plots=False)
    rows.append({
        "eksperimen": name,
        "imgsz": imgsz,
        "mAP50": float(met.box.map50),
        "mAP50-95": float(met.box.map),
        "precision": float(met.box.mp),
        "recall": float(met.box.mr),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
df.to_csv(WORK / "hasil_praktik_val.csv", index=False)

best_row = df.loc[df["mAP50-95"].idxmax()]
print("\n>>> Model terbaik (VAL mAP50-95):", best_row["eksperimen"], best_row["mAP50-95"])

Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 121 layers, 2,397,025 parameters, 0 gradients, 5.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 536.1±190.1 MB/s, size: 16.2 KB)
val: Scanning /content/pj1/dataset_v1/labels/val.cache... 278 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 278/278 89.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 9.5it/s 1.9s
                   all        278        278      0.964      0.925      0.984      0.874
                Blue 3         10         10      0.898          1      0.995      0.833
                Blue 8         10         10      0.964          1      0.995      0.942
                Blue D          8          8      0.952      0.875      0.982       0.88
       Blue Left Arrow         10         10          1      0.967      0.995      0.931
                Blue S          5        

In [9]:
best_name = best_row["eksperimen"]
best_imgsz = int(best_row["imgsz"])
best_pt = WORK / "hasil_praktik" / best_name / "weights" / "best.pt"

model = YOLO(str(best_pt))
met_test = model.val(data=DATA_YAML, split="test", imgsz=best_imgsz, plots=True)
print("TEST mAP50:   ", float(met_test.box.map50))
print("TEST mAP50-95:", float(met_test.box.map))

Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 121 layers, 2,397,025 parameters, 0 gradients, 5.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 493.0±189.1 MB/s, size: 16.6 KB)
val: Scanning /content/pj1/dataset_v1/labels/test... 140 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 140/140 2.1Kit/s 0.1s
val: New cache created: /content/pj1/dataset_v1/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 2.9it/s 3.1s
                   all        140        140      0.924      0.971      0.994      0.872
                Blue 3          7          7      0.994          1      0.995      0.875
                Blue 8          5          5          1      0.937      0.995       0.95
                Blue D          5          5       0.93          1      0.995      0.915
       Blue Left Arrow          3          3          1      0.936      

In [10]:
import numpy as np

def latency_ms(weights, imgsz, n=50):
    m = YOLO(str(weights))
    dummy = np.random.randint(0, 255, (imgsz, imgsz, 3), dtype=np.uint8)
    for _ in range(10):
        m.predict(dummy, imgsz=imgsz, verbose=False)
    times = []
    for _ in range(n):
        t0 = time.perf_counter()
        m.predict(dummy, imgsz=imgsz, verbose=False)
        times.append((time.perf_counter() - t0) * 1000)
    return sum(times) / len(times)

for name, imgsz in [("E1_pretrained", 416), ("E3_tuning", 640)]:
    w = WORK / "hasil_praktik" / name / "weights" / "best.pt"
    if w.exists():
        ms = latency_ms(w, imgsz)
        print(f"{name:16s} imgsz={imgsz}  {ms:.1f} ms/citra  ({1000/ms:.1f} FPS)")

E1_pretrained    imgsz=416  11.5 ms/citra  (87.3 FPS)
E3_tuning        imgsz=640  10.7 ms/citra  (93.9 FPS)


In [11]:
out = WORK / "kumpul_pj1"
if out.exists():
    shutil.rmtree(out)
out.mkdir()

# salin results.png + best.pt tiap eksperimen
for name in ["E1_pretrained", "E2_scratch", "E3_tuning"]:
    src = WORK / "hasil_praktik" / name
    if not src.exists():
        continue
    dst = out / name
    dst.mkdir()
    for f in ["results.png", "confusion_matrix.png"]:
        p = src / f
        if p.exists():
            shutil.copy(p, dst / f)
    bp = src / "weights" / "best.pt"
    if bp.exists():
        (dst / "weights").mkdir(exist_ok=True)
        shutil.copy(bp, dst / "weights" / "best.pt")

if (WORK / "hasil_praktik_val.csv").exists():
    shutil.copy(WORK / "hasil_praktik_val.csv", out / "hasil_praktik_val.csv")

shutil.make_archive(str(WORK / "kumpul_pj1"), "zip", out)
from google.colab import files
files.download(str(WORK / "kumpul_pj1.zip"))
print("Download kumpul_pj1.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download kumpul_pj1.zip


In [ ]:
from ultralytics import YOLO
from pathlib import Path

# Ganti ke model terbaik kamu
WEIGHTS = "hasil_praktik/E3_tuning/weights/best.pt"  # atau E1_pretrained/...
model = YOLO(WEIGHTS)

# 1) Satu gambar
model.predict(
    source="dataset_v1/images/test/NAMA_GAMBAR.jpg",  # ganti path
    save=True,
    project="prediksi",
    name="satu_gambar",
    conf=0.25,
    imgsz=640,  # samakan dengan eksperimen (416 untuk E1)
)

# 2) Seluruh folder test (banyak gambar)
model.predict(
    source="dataset_v1/images/test",
    save=True,
    project="prediksi",
    name="test_set",
    conf=0.25,
    imgsz=640,
)

print("Hasil gambar ada di folder: prediksi/test_set/")

In [18]:
from ultralytics import YOLO
from pathlib import Path

# Ganti ke model terbaik kamu
WEIGHTS = "/content/pj1/hasil_praktik/E3_tuning/weights/best.pt"  # atau E1_pretrained/...
model = YOLO(WEIGHTS)

# 1) Satu gambar
model.predict(
    source="/content/pj1/dataset_v1/images/test/test_20220817_093535_jpg.rf.8e5c78f5473de79980baf2859e730d3f.jpg",  # ganti path
    save=True,
    project="prediksi",
    name="satu_gambar",
    conf=0.25,
    imgsz=640,  # samakan dengan eksperimen (416 untuk E1)
)

# 2) Seluruh folder test (banyak gambar)
model.predict(
    source="/content/pj1/dataset_v1/images/test",
    save=True,
    project="prediksi",
    name="test_set",
    conf=0.25,
    imgsz=640,
)

print("Hasil gambar ada di folder: prediksi/test_set/")


image 1/1 /content/pj1/dataset_v1/images/test/test_20220817_093535_jpg.rf.8e5c78f5473de79980baf2859e730d3f.jpg: 640x640 1 Blue D, 10.4ms
Speed: 2.0ms preprocess, 10.4ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/detect/prediksi/satu_gambar-2

image 1/140 /content/pj1/dataset_v1/images/test/test_20220817_093535_jpg.rf.8e5c78f5473de79980baf2859e730d3f.jpg: 640x640 1 Blue D, 11.7ms
image 2/140 /content/pj1/dataset_v1/images/test/test_20220817_093851_jpg.rf.17fd2ce32e23956e371bd35ba0d3b018.jpg: 640x640 1 Yellow U, 10.4ms
image 3/140 /content/pj1/dataset_v1/images/test/test_IMG_2510_JPG.rf.39e2e5a59b1347d72649c443d72aac3d.jpg: 640x640 1 Yellow 5, 10.3ms
image 4/140 /content/pj1/dataset_v1/images/test/test_IMG_2528_JPG.rf.46baf4f93b7465505d52e37f7e9cdae3.jpg: 640x640 1 Green W, 10.4ms
image 5/140 /content/pj1/dataset_v1/images/test/test_image132_jpg.rf.99f7e95a98cd613b6d3b8cabb09932b7.jpg: 640x640 1 Blue Left Arrow, 10.4ms
image 6/140 /co